UsageError: %%writefile is a cell magic, but the cell body is empty.


In [16]:
pip install -q groq

Note: you may need to restart the kernel to use updated packages.


In [17]:
pip install requests

In [34]:
%%writefile app.py
import json 
import requests 
from groq import Groq
from dotenv import load_dotenv
from pprint import pprint
load_dotenv()
import os

api_key = os.getenv("Groq_API_KEY")
client=Groq(api_key=api_key)

def get_weather(location):
  api_key = os.getenv("Weather_API_KEY")
  url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"

  response = requests.get(url)
  data = response.json()
  if data.get("cod") == 200:
    return json.dumps({
      "location": location,
      "temperature": data["main"]["temp"],
      "description": data["weather"][0]["description"]
    })
  else:
    return json.dumps({"Oops! Something went wrong."})


tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name like Mumbai, London"
          }},
        "required": ["location"]
      }}}
]




llm_messages = [
  {
    "role": "system",
    "content": "You are a weather assistant. Use get_weather function when asked about weather."
  },
  {
    "role": "user",
    "content": "What's the weather in Mumbai?"
  }
]

response = client.chat.completions.create(
  model="llama-3.3-70b-versatile",
  messages=llm_messages,
  tools=tools,
  tool_choice="auto"
)

response_message = response.choices[0].message

if response_message.tool_calls:
  tool_call = response_message.tool_calls[0]
  arguments = json.loads(tool_call.function.arguments)
  location = arguments['location']
  weather_data = get_weather(location)

  llm_messages.append(response_message)


  llm_messages.append({
      "role": "tool",
      "tool_call_id": tool_call.id,
      "content": json.dumps(weather_data)
  })



final_response = client.chat.completions.create(
      messages = llm_messages,
      model = "llama-3.3-70b-versatile",
      tools = tools,
      tool_choice = "auto"
  )

print(final_response.choices[0].message.content)



Writing app.py


In [ ]:
import requests
import json
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
  api_key=os.getenv("GROQ_API_KEY")
)

def get_weather(location):
   """Get weather for any city"""
   api_key = os.getenv("OPEN_WEATHER_API_KEY")
   url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"

   response = requests.get(url)
   data = response.json()

   if data["cod"] == 200:
       return {
           "location": location,
           "temperature": data["main"]["temp"],
           "description": data["weather"][0]["description"]
       }
   else:
       return {"error": "City not found"}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name like Mumbai, London"
                    }
                },
                "required": ["location"]
            }
        }
    }
]
messages = [
    {
        "role": "system",
        "content": "You are a weather assistant. Use get_weather function when asked about weather."
    },
    {
        "role": "user",
        "content": "What's the weather in Hyd?"
    }
]
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools,
    tool_choice="auto"   
)
print(response)
response_message = response.choices[0].message

if response_message.tool_calls:
    print("Model wants to use a tool!")
else:
    print("Model answered directly")

if response_message.tool_calls:
    tool_call = response_message.tool_calls[0]

    arguments = json.loads(tool_call.function.arguments)
    location = arguments["location"]
    print(f"LLM wants weather for: {location}")

    weather_data = get_weather(location)
    print(f"Weather data: {weather_data}")
    messages.append(response_message)

    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(weather_data)
    })
final_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )

print(final_response.choices[0].message.content)


The current weather in Mumbai is clear sky with a temperature of 30.49 degrees.
